# 01 - Data Extraction

Compiled version of the extraction notebook.

Design choices:
- Core universe: Gold, Brent, DXY, VIX, US10Y. Copper removed — it served no role in the primary analysis.
- Start date set to `2007-07-01` — rationale in the configuration cell below.
- Use explicit ticker-to-asset mapping instead of positional column renaming.
- Save raw yfinance output and a clean close-price panel for Step 02.

## Reader Orientation

This notebook starts the nowcasting workflow. Its job is to create a clean market data universe where Gold can later be evaluated as a real-time instability indicator for a Brent crude risk book. The dataset runs from July 2007 — the earliest date where all five core assets have yfinance coverage — giving the pipeline access to the GFC after the z-score burn-in period clears.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import yfinance as yf

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
TICKERS = {
    "Gold":  "GC=F",       # Gold futures; primary alarm asset
    "Brent": "BZ=F",       # Brent crude futures; primary risk-book exposure
    "DXY":   "DX-Y.NYB",   # US Dollar Index
    "VIX":   "^VIX",       # CBOE Volatility Index
    "US10Y": "^TNX",       # 10Y Treasury yield
}

# BZ=F (Brent) is the binding constraint: yfinance data starts 2007-07-30.
# Setting start to 2007-07-01 captures that first row without downloading
# months of other-asset data that strict dropna would immediately discard.
# After a 252-day z-score burn-in (~mid-2008), the evaluation window covers
# the GFC peak: oil at $147 (July 2008), Lehman collapse (Sep 2008), and the
# full 2008-2009 credit crisis — the most significant commodity stress episode
# excluded by the previous 2010 start date.
START_DATE = "2007-07-01"
END_DATE = None
PRICE_FIELD = "Close"

TICKER_TO_ASSET = {ticker: asset for asset, ticker in TICKERS.items()}
TICKERS

{'Gold': 'GC=F',
 'Brent': 'BZ=F',
 'DXY': 'DX-Y.NYB',
 'VIX': '^VIX',
 'US10Y': '^TNX'}

### Why This Asset Set

Gold is the alarm asset. Brent is the primary commodity risk-book exposure. DXY, VIX, and US10Y are conditioning variables used to interpret what gold is reacting to — dollar stress, equity fear, and rate moves respectively. Copper was removed: it was designated robustness-only from the start and never contributed to any core result, so carrying it through the pipeline added complexity without payoff.

In [3]:
raw_data = yf.download(
    tickers=list(TICKERS.values()),
    start=START_DATE,
    end=END_DATE,
    auto_adjust=False,
    group_by="column",
    threads=True,
)

raw_data.tail()


[*********************100%***********************]  5 of 5 completed


Price       Adj Close                                                Close  \
Ticker           BZ=F   DX-Y.NYB         GC=F   ^TNX       ^VIX       BZ=F   
Date                                                                         
2026-06-01  94.980003  99.199997  4475.200195  4.475  16.049999  94.980003   
2026-06-02  96.000000  99.220001  4489.100098  4.455  15.770000  96.000000   
2026-06-03  97.809998  99.529999  4436.700195  4.491  16.059999  97.809998   
2026-06-04  95.029999  99.410004  4475.799805  4.477  15.400000  95.029999   
2026-06-05  94.889999  99.203003  4495.100098    NaN  15.750000  94.889999   

Price                                                 ...       Open  \
Ticker       DX-Y.NYB         GC=F   ^TNX       ^VIX  ...       BZ=F   
Date                                                  ...              
2026-06-01  99.199997  4475.200195  4.475  16.049999  ...  92.410004   
2026-06-02  99.220001  4489.100098  4.455  15.770000  ...  95.400002   
2026-06-03  99.529999  4436.700195  4.491  16.059999  ...  96.070000   
2026-06-04  99.410004  4475.799805  4.477  15.400000  ...  97.430000   
2026-06-05  99.203003  4495.100098    NaN  15.750000  ...  95.070000   

Price                                                  Volume           \
Ticker       DX-Y.NYB         GC=F   ^TNX       ^VIX     BZ=F DX-Y.NYB   
Date                                                                     
2026-06-01  98.959999  4523.500000  4.457  15.880000  57796.0      0.0   
2026-06-02  99.169998  4488.000000  4.430  16.280001  40210.0      0.0   
2026-06-03  99.290001  4471.200195  4.485  15.970000  40867.0      0.0   
2026-06-04  99.459999  4447.399902  4.457  16.330000  40867.0      0.0   
2026-06-05  99.431999  4503.000000    NaN  15.870000  13367.0      0.0   

Price                          
Ticker         GC=F ^TNX ^VIX  
Date                           
2026-06-01    835.0  0.0  0.0  
2026-06-02    456.0  0.0  0.0  
2026-06-03   2913.0  0.0  0.0  
2026-06-04   2913.0  0.0  0.0  
2026-06-05  47066.0  NaN  0.0  

[5 rows x 30 columns]

In [4]:
def extract_price_panel(raw: pd.DataFrame, price_field: str = "Close") -> pd.DataFrame:
    if isinstance(raw.columns, pd.MultiIndex):
        fields = raw.columns.get_level_values(0)
        if price_field in fields:
            prices = raw[price_field].copy()
        elif "Adj Close" in fields:
            prices = raw["Adj Close"].copy()
        else:
            raise ValueError(f"Could not find {price_field!r} or 'Adj Close' in yfinance output")
    else:
        prices = raw.copy()

    prices = prices.rename(columns=TICKER_TO_ASSET)
    prices = prices[list(TICKERS.keys())]
    prices.index = pd.to_datetime(prices.index)
    return prices.sort_index()


prices = extract_price_panel(raw_data, PRICE_FIELD)
prices.tail()


Ticker,Gold,Brent,DXY,VIX,US10Y
Date,,,,,
2026-06-01,4475.200195,94.980003,99.199997,16.049999,4.475
2026-06-02,4489.100098,96.000000,99.220001,15.770000,4.455
2026-06-03,4436.700195,97.809998,99.529999,16.059999,4.491
2026-06-04,4475.799805,95.029999,99.410004,15.400000,4.477
2026-06-05,4495.100098,94.889999,99.203003,15.750000,NaN


In [5]:
raw_path = RAW_DIR / "yfinance_raw_data.parquet"
prices_path = RAW_DIR / "prices_close_raw.parquet"

raw_data.to_parquet(raw_path)
prices.to_parquet(prices_path)

print("Saved raw data to:", raw_path)
print("Saved close-price panel to:", prices_path)
print("Assets:", list(TICKERS))
print("Shape:", prices.shape)
print("Date range:", prices.index.min(), "to", prices.index.max())
print("Missing values:")
print(prices.isna().sum())

Saved raw data to: C:\Users\sohwe\Desktop\SMU\MQF\Commodities Risk Management\qf637\data\raw\yfinance_raw_data.parquet
Saved close-price panel to: C:\Users\sohwe\Desktop\SMU\MQF\Commodities Risk Management\qf637\data\raw\prices_close_raw.parquet
Assets: ['Gold', 'Brent', 'DXY', 'VIX', 'US10Y']
Shape: (4769, 5)
Date range: 2007-07-02 00:00:00 to 2026-06-05 00:00:00
Missing values:
Ticker
Gold      7
Brent    77
DXY       5
VIX       5
US10Y     9
dtype: int64


### Result Comment And Significance

The dataset now opens in July 2007, with aligned data from all five assets beginning on Brent's first available date (2007-07-30). After a 252-day z-score burn-in, the evaluation window starts around mid-2008 — covering the GFC peak, the 2008-2009 credit crisis, and all subsequent stress episodes through to 2026.

Missing values are expected to remain small. Brent typically has the most, driven by calendar mismatches between NYMEX and other exchanges. The strict dropna in Step 02 handles these without forward-filling, which would otherwise introduce artificial zero returns into the signal.